In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-22'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 75.0]}}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.2177, -0.1756, -0.4150, -0.2463, -1.2420, -0.0019,  0.3806,  0.5169,
          0.1421, -0.0152, -0.8218, -0.0747]], device='cuda:0')
Scaled actions :  tensor([[-0.2177, -0.1756, -0.4150, -0.2463, -1.2420, -0.0019,  0.3806,  0.5169,
          0.1421, -0.0152, -0.8218, -0.0747]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -2.1768e-01, -1.7564e-01,
         -4.1498e-01, -2.4629e-01, -1.2420e+00, -1.9328e-03,  3.8062e-01,
          5.1691e-01,  1.4209e-01, -1.5228e-02, -8.2180e-01, -7.4657e-02]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.2407, -0.7349, -0.0653,  0.7013, -1.4460, -0.1905,  0.2373,  0.8060,
          0.1606,  0.1547, -0.8579,  0.0533]], device='cuda:0')
Scaled actions :  tensor([[-0.2407, -0.7349, -0.0653,  0.7013, -1.4460, -0.1905,  0.2373,  0.8060,
          0.1606,  0.1547, -0.8579,  0.0533]], device='cuda:0')
obs :  tensor([[ 2.2340e-02, -8.8129e-02, -2.8685e-01, -1.7498e-03, -3.5102e-04,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -3.8650e-02,
         -4.0750e-03, -8.7845e-04,  1.2010e-02, -8.5629e-02,  2.9852e-03,
          4.6567e-02,  6.3537e-03,  5.2298e-03,  1.3094e-02, -9.7825e-02,
         -1.6203e-02, -3.0669e-01, -3.9268e-02, -1.0653e-03,  8.6106e-02,
         -7.7232e-01,  2.1306e-02,  4.3912e-01,  5.2114e-02,  6.7609e-02,
          8.0966e-02, -8.7720e-01, -8.4111e-02, -2.4074e-01, -7.3488e-01,
         -6.5269e-02,  7.0128e-01, -1.4460e+00, -1.9047e-01,  2.3733e-01,
          8.0597e-01,  1.6062e-01,  1.5471e-01, -8.5787e-01,  5.3

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.5871,  0.0648, -0.2719, -0.2291, -0.2344, -0.0208, -0.9227,  0.2908,
         -0.3624, -0.8908, -0.3034,  0.1912]], device='cuda:0')
Scaled actions :  tensor([[ 0.5871,  0.0648, -0.2719, -0.2291, -0.2344, -0.0208, -0.9227,  0.2908,
         -0.3624, -0.8908, -0.3034,  0.1912]], device='cuda:0')
obs :  tensor([[ 0.1324, -0.3356, -0.3153, -0.0110, -0.0039, -0.9999,  1.0000,  0.0000,
          0.0000, -0.1121, -0.0216, -0.0104,  0.0611, -0.3412, -0.0506,  0.1235,
          0.0228,  0.0281,  0.0400, -0.3194,  0.0049, -0.3068, -0.1422, -0.0751,
          0.3610, -1.6876, -0.2947,  0.2715,  0.1172,  0.1585,  0.1562, -1.1559,
          0.1000,  0.5871,  0.0648, -0.2719, -0.2291, -0.2344, -0.0208, -0.9227,
          0.2908, -0.3624, -0.8908, -0.3034,  0.1912]], device='cuda:0')
torques: [ -47.51786226  200.          121.84677279   83.40941238   61.46041762
   -3.61973368   55.64255747 -200.          -42.43948139  200.
 -200.           11.68093482]
データ収集:

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.4191,  0.4616,  0.0312, -0.6659,  0.1956, -0.2930, -0.3043,  0.0192,
         -0.8608, -1.1204,  0.3782, -0.5254]], device='cuda:0')
Scaled actions :  tensor([[ 0.4191,  0.4616,  0.0312, -0.6659,  0.1956, -0.2930, -0.3043,  0.0192,
         -0.8608, -1.1204,  0.3782, -0.5254]], device='cuda:0')
obs :  tensor([[-0.4230,  0.3469, -0.3340, -0.0090,  0.0033, -1.0000,  1.0000,  0.0000,
          0.0000, -0.1203, -0.0269, -0.0443,  0.1115, -0.5703, -0.0493,  0.1273,
          0.0705,  0.0336,  0.0590, -0.4419,  0.0602,  0.1701,  0.0642, -0.2430,
          0.1668, -0.7029,  0.0606, -0.1650,  0.3128, -0.0908,  0.0999, -0.1622,
          0.2191,  0.4191,  0.4616,  0.0312, -0.6659,  0.1956, -0.2930, -0.3043,
          0.0192, -0.8608, -1.1204,  0.3782, -0.5254]], device='cuda:0')
torques: [-200.          200.         -200.         -200.          200.
   63.58485326  200.          139.72791838 -200.         -200.
  200.           -2.60461481]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-1.4022, -0.9780,  1.2066,  0.7074,  0.0367, -0.1629,  0.7488, -0.9236,
          0.8490, -0.1205, -0.4834, -0.1575]], device='cuda:0')
Scaled actions :  tensor([[-1.4022, -0.9780,  1.2066,  0.7074,  0.0367, -0.1629,  0.7488, -0.9236,
          0.8490, -0.1205, -0.4834, -0.1575]], device='cuda:0')
obs :  tensor([[-0.5764,  0.5148, -0.1645,  0.0089,  0.0238, -0.9997,  1.0000,  0.0000,
          0.0000, -0.0402, -0.0060, -0.0785,  0.1099, -0.6065, -0.1091,  0.0500,
          0.1362,  0.0145,  0.0617, -0.3657, -0.0026,  0.5903,  0.1234, -0.1077,
         -0.1533,  0.2451, -0.3956, -0.5189,  0.3381, -0.0873, -0.0577,  0.8256,
         -0.7486, -1.4022, -0.9780,  1.2066,  0.7074,  0.0367, -0.1629,  0.7488,
         -0.9236,  0.8490, -0.1205, -0.4834, -0.1575]], device='cuda:0')
torques: [-200.         -200.         -200.         -200.          200.
 -200.          200.          200.          200.         -200.
  200.           21.22360907]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-1.2696, -0.6119,  0.2147,  0.8531, -0.9830,  0.5255,  1.2080, -0.3680,
         -0.3397, -0.3392, -1.5119,  0.5170]], device='cuda:0')
Scaled actions :  tensor([[-1.2696, -0.6119,  0.2147,  0.8531, -0.9830,  0.5255,  1.2080, -0.3680,
         -0.3397, -0.3392, -1.5119,  0.5170]], device='cuda:0')
obs :  tensor([[ 1.5679e-01, -2.3107e-01, -1.7689e-01,  1.3483e-02,  3.1169e-02,
         -9.9942e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.8209e-02,
         -9.7746e-03, -8.0829e-02,  9.6968e-02, -4.6443e-01, -1.3559e-01,
          8.8115e-04,  1.6583e-01,  2.6625e-02,  3.9748e-02, -3.0908e-01,
         -8.6571e-02,  1.4004e-01, -1.2569e-01,  8.4810e-02,  6.2257e-03,
          9.7401e-01, -5.8386e-02, -1.9929e-02, -1.9828e-02,  1.2444e-01,
         -4.3224e-02, -1.4790e-01, -2.4803e-01, -1.2696e+00, -6.1191e-01,
          2.1474e-01,  8.5310e-01, -9.8305e-01,  5.2553e-01,  1.2080e+00,
         -3.6802e-01, -3.3966e-01, -3.3916e-01, -1.5119e+00,  5.1

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.2464,  0.3062, -0.7449, -0.2528, -1.2072, -0.1726, -0.2525,  1.0384,
         -0.3544, -0.2416, -0.6706,  0.1792]], device='cuda:0')
Scaled actions :  tensor([[ 0.2464,  0.3062, -0.7449, -0.2528, -1.2072, -0.1726, -0.2525,  1.0384,
         -0.3544, -0.2416, -0.6706,  0.1792]], device='cuda:0')
obs :  tensor([[ 0.8438, -0.2729,  0.0314,  0.0032,  0.0098, -0.9999,  1.0000,  0.0000,
          0.0000,  0.0065, -0.0623, -0.0599,  0.1137, -0.3781, -0.0389,  0.0453,
          0.1321,  0.0445,  0.0465, -0.4197, -0.0416, -0.3088, -0.3812,  0.1136,
          0.1424, -0.0119,  0.9255,  0.4195, -0.2880,  0.0700,  0.0878, -0.8925,
          0.5992,  0.2464,  0.3062, -0.7449, -0.2528, -1.2072, -0.1726, -0.2525,
          1.0384, -0.3544, -0.2416, -0.6706,  0.1792]], device='cuda:0')
torques: [ 200.         -200.         -200.         -200.         -200.
  186.60090111 -200.         -200.          200.          200.
 -200.          200.        ]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.6052,  0.9202, -0.1561, -1.1642, -0.2429, -1.0048, -0.2938,  1.1529,
         -0.0334, -0.1547,  0.1175, -0.7155]], device='cuda:0')
Scaled actions :  tensor([[ 0.6052,  0.9202, -0.1561, -1.1642, -0.2429, -1.0048, -0.2938,  1.1529,
         -0.0334, -0.1547,  0.1175, -0.7155]], device='cuda:0')
obs :  tensor([[ 2.2249e-01,  4.0342e-01,  5.4391e-04,  5.6272e-03, -9.4204e-03,
         -9.9994e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.9883e-03,
         -1.1322e-01, -5.3475e-02,  1.2057e-01, -4.8147e-01,  3.8248e-02,
          8.0840e-02,  1.0402e-01,  4.1000e-02,  5.6690e-02, -5.3411e-01,
          3.9961e-02,  1.0770e-01, -1.4511e-01, -2.8075e-02, -4.8975e-02,
         -7.9001e-01, -5.5302e-02, -1.3691e-02, -3.1152e-02, -1.0288e-01,
          2.9434e-02, -2.8777e-01,  2.7029e-01,  6.0521e-01,  9.2018e-01,
         -1.5609e-01, -1.1642e+00, -2.4287e-01, -1.0048e+00, -2.9379e-01,
          1.1529e+00, -3.3378e-02, -1.5467e-01,  1.1748e-01, -7.1

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-1.3408,  0.0663,  0.9456,  0.3654,  0.1469,  0.3257,  0.5116, -0.3749,
          1.0874,  0.5916, -0.0759, -0.0833]], device='cuda:0')
Scaled actions :  tensor([[-1.3408,  0.0663,  0.9456,  0.3654,  0.1469,  0.3257,  0.5116, -0.3749,
          1.0874,  0.5916, -0.0759, -0.0833]], device='cuda:0')
obs :  tensor([[-0.3708,  0.7400, -0.1228,  0.0314, -0.0061, -0.9995,  1.0000,  0.0000,
          0.0000,  0.0590, -0.1176, -0.0640,  0.0896, -0.5315, -0.0809,  0.0277,
          0.1278,  0.0163,  0.0481, -0.4856, -0.0224,  0.5305,  0.0712, -0.0667,
         -0.2432,  0.1919, -1.0388, -0.4765,  0.2520, -0.1135, -0.1183,  0.6733,
         -0.7810, -1.3408,  0.0663,  0.9456,  0.3654,  0.1469,  0.3257,  0.5116,
         -0.3749,  1.0874,  0.5916, -0.0759, -0.0833]], device='cuda:0')
torques: [-200.          200.           21.10313135 -200.          200.
 -200.          200.          200.         -189.27670751 -200.
  200.         -200.        ]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-1.2339, -0.3668,  0.8811,  0.8332, -0.6981,  0.9092,  1.2353, -0.7873,
          0.4689,  0.3318, -0.5249,  0.5655]], device='cuda:0')
Scaled actions :  tensor([[-1.2339, -0.3668,  0.8811,  0.8332, -0.6981,  0.9092,  1.2353, -0.7873,
          0.4689,  0.3318, -0.5249,  0.5655]], device='cuda:0')
obs :  tensor([[-2.7633e-01, -6.0422e-02, -1.2240e-01,  4.2716e-02,  8.0720e-03,
         -9.9905e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.0280e-01,
         -8.3998e-02, -8.2298e-02,  9.3554e-02, -4.1091e-01, -1.5295e-01,
         -1.5674e-02,  1.6676e-01,  1.6671e-02,  3.8927e-02, -3.4815e-01,
         -1.0035e-01, -6.1712e-04,  2.0022e-01, -4.9501e-02,  1.3564e-01,
          8.5811e-01,  1.0900e-01, -1.0357e-02,  1.2241e-01,  8.0793e-02,
          1.5086e-02,  5.5161e-01,  5.3200e-03, -1.2339e+00, -3.6685e-01,
          8.8112e-01,  8.3317e-01, -6.9811e-01,  9.0923e-01,  1.2353e+00,
         -7.8732e-01,  4.6887e-01,  3.3181e-01, -5.2494e-01,  5.

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.6526, -1.2541, -0.8536, -1.1139, -1.5270, -0.5701,  0.0826,  0.8367,
         -0.7764, -0.5012, -0.3955,  0.1160]], device='cuda:0')
Scaled actions :  tensor([[ 0.6526, -1.2541, -0.8536, -1.1139, -1.5270, -0.5701,  0.0826,  0.8367,
         -0.7764, -0.5012, -0.3955,  0.1160]], device='cuda:0')
obs :  tensor([[ 0.3697, -0.7262, -0.1278,  0.0256,  0.0051, -0.9997,  1.0000,  0.0000,
          0.0000,  0.0583, -0.0686, -0.0713,  0.1402, -0.3477, -0.0229,  0.0292,
          0.1586,  0.0496,  0.0554, -0.3439,  0.0119, -0.4053, -0.0343,  0.1427,
          0.3117, -0.1265,  1.0921,  0.4162, -0.1645,  0.2373,  0.1344, -0.3444,
          1.0162,  0.6526, -1.2541, -0.8536, -1.1139, -1.5270, -0.5701,  0.0826,
          0.8367, -0.7764, -0.5012, -0.3955,  0.1160]], device='cuda:0')
torques: [ 200.         -200.          200.          200.          -64.07516012
  200.         -200.         -200.          200.          200.
 -200.          200.        ]
データ収集

In [29]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.063, Scaled action max=1.063
Step 1/100, Total steps: 512
steps: 512
actions : tensor([[-0.6888, -0.9977,  0.7766, -1.2607,  1.0625,  0.3158,  0.5177, -1.1010,
         -0.6955, -0.8052, -0.1879, -0.2625]], device='cuda:0')
target_dof_pos: tensor([[-0.3037, -0.5302,  0.0282,  2.0690,  0.9841,  0.3275, -0.1617, -0.9761,
         -2.4547,  1.7359, -1.8051, -0.1694]], device='cuda:0')
Step 1: Original action max=1.087, Scaled action max=1.087
Step 2: Original action max=0.855, Scaled action max=0.855
Step 21/100, Total steps: 532
steps: 532
actions : tensor([[ 0.9753, -0.5855, -0.5389, -0.3010, -0.2880, -0.3812, -0.1389, -0.4246,
          0.4604, -0.9016,  0.1023, -0.7179]], device='cuda:0')
target_dof_pos: tensor([[ 0.1157, -0.7234, -1.0995,  2.5961, -1.4391, -0.3342,  1.2215, -1.3126,
         -1.7671,  1.7119, -1.7564,  0.0779]], device='cuda:0')
Step 41/100, Total steps: 552
steps: 552
actions : tensor([[ 1.0912,  0.8966, -0.6145, -0.9428,  0.5293, -0.49

In [30]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [31]:
env.sim.stop()

In [31]:
env.reset()
cnt = 0

In [61]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-14_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (362, 58)
